# LeetCode 1488 — Avoid Flood in The City bằng Python

Link đề: https://leetcode.com/problems/avoid-flood-in-the-city/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Binary Search, Greedy, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def avoidFlood(self, rains: list[int]) -> list[int]:
        ...
```

## 1. Mục tiêu học

1. Mô hình hoá bài toán mô phỏng theo ngày thành bài **gán tài nguyên (ngày khô) cho yêu cầu (hồ sắp bị mưa lần hai)** có ràng buộc thời gian.
2. Hiểu ý tưởng **trì hoãn quyết định** (lazy): không quyết định hồ nào được làm khô vào ngày khô ngay lúc đó, mà để dành, chỉ "tiêu" ngày khô khi thật sự cần.
3. Dùng `bisect` trên danh sách ngày khô đã sắp xếp để tìm **ngày khô sớm nhất sau một mốc**.
4. Biết hai cách khác: **heap theo hạn chót** (Earliest Deadline First) và **DSU "ngày trống kế tiếp"**.
5. Kiểm thử bài **nhiều đáp án** bằng hàm **mô phỏng kiểm tra tính hợp lệ**, và kiểm tra trường hợp `[]` bằng vét cạn.

## 2. Đề bài & ví dụ

Có `10^9` hồ, ban đầu đều rỗng. Mảng `rains`:
- `rains[i] > 0`: ngày `i` trời mưa xuống hồ `rains[i]`. Hồ đang rỗng ⇒ đầy; hồ **đang đầy** ⇒ **lụt**.
- `rains[i] == 0`: ngày `i` không mưa, bạn **phải chọn một hồ** để làm khô (hồ đầy ⇒ rỗng; hồ rỗng ⇒ không đổi).

Trả về `ans` cùng độ dài: `ans[i] = -1` nếu `rains[i] > 0`; `ans[i]` = hồ được chọn làm khô nếu `rains[i] == 0`. Nhiều đáp án ⇒ trả về bất kỳ. Không thể tránh lụt ⇒ trả `[]`.

**Ví dụ 1:** `[1,2,3,4]` → `[-1,-1,-1,-1]` (không có ngày khô, cũng không lụt).

**Ví dụ 2:** `[1,2,0,0,2,1]` → `[-1,-1,2,1,-1,-1]` (cũng chấp nhận `[-1,-1,1,2,-1,-1]`).

**Ví dụ 3:** `[1,2,0,1,2]` → `[]` (chỉ một ngày khô mà cả hồ 1 và 2 đều bị mưa lại ⇒ chắc chắn một hồ lụt).

**Ràng buộc:** `1 <= len(rains) <= 10^5`, `0 <= rains[i] <= 10^9`.

In [1]:
import bisect
import heapq
import random
import time
from typing import List

examples = [
    ([1, 2, 3, 4], [-1, -1, -1, -1]),
    ([1, 2, 0, 0, 2, 1], [-1, -1, 2, 1, -1, -1]),
    ([1, 2, 0, 1, 2], []),
]
for rains, want in examples:
    print(rains, "-> một đáp án:", want)

[1, 2, 3, 4] -> một đáp án: [-1, -1, -1, -1]
[1, 2, 0, 0, 2, 1] -> một đáp án: [-1, -1, 2, 1, -1, -1]
[1, 2, 0, 1, 2] -> một đáp án: []


Vì có nhiều đáp án, ta cần một **hàm kiểm tra** `is_valid(rains, ans)` mô phỏng lại từng ngày: ngày mưa phải có `ans[i] == -1` và hồ không được đang đầy; ngày khô phải có `ans[i] >= 1` (là một hồ) và làm khô hồ đó.

In [2]:
def is_valid(rains, ans):
    if len(ans) != len(rains):
        return False
    full = set()
    for r, a in zip(rains, ans):
        if r > 0:
            if a != -1 or r in full:
                return False
            full.add(r)
        else:
            if not isinstance(a, int) or a < 1:
                return False
            full.discard(a)
    return True

print(is_valid(*examples[0]), is_valid(*examples[1]), is_valid([1, 2, 0, 0, 2, 1], [-1, -1, 1, 2, -1, -1]))
print(is_valid([1, 2, 0, 1, 2], [-1, -1, 1, -1, -1]), "(hồ 2 lụt)")

True True True
False (hồ 2 lụt)


## 3. Kiến thức nền

### 3.1 `set` các hồ đang đầy và `dict` ngày mưa gần nhất

- `full`: tập hồ đang đầy.
- `last[lake]`: ngày gần nhất trời mưa xuống `lake` (khi hồ đang đầy, đó là ngày nó **bắt đầu đầy**).

In [3]:
rains = [1, 2, 0, 0, 2, 1]
last = {}
for i, r in enumerate(rains):
    if r:
        print(f"ngày {i}: mưa hồ {r}, lần mưa trước: {last.get(r, 'chưa có')}")
        last[r] = i

ngày 0: mưa hồ 1, lần mưa trước: chưa có
ngày 1: mưa hồ 2, lần mưa trước: chưa có
ngày 4: mưa hồ 2, lần mưa trước: 1
ngày 5: mưa hồ 1, lần mưa trước: 0


### 3.2 `bisect` trên danh sách đã sắp xếp

`bisect.bisect_right(a, x)` trả vị trí đầu tiên có phần tử `> x`. Nếu `a` là danh sách các ngày khô (tăng dần), thì `a[bisect_right(a, x)]` là **ngày khô sớm nhất sau ngày `x`**. `a.pop(j)` xoá phần tử đó.

In [4]:
dry = [2, 3, 7, 9]
for x in [0, 3, 5, 9]:
    j = bisect.bisect_right(dry, x)
    print(f"x={x}: j={j} ->", dry[j] if j < len(dry) else "không còn ngày khô nào sau x")

x=0: j=0 -> 2
x=3: j=2 -> 7
x=5: j=2 -> 7
x=9: j=4 -> không còn ngày khô nào sau x


### 3.3 Min-heap với `heapq`

`heapq.heappush(h, (khóa, dữ liệu))` và `heapq.heappop(h)` lấy phần tử có khoá **nhỏ nhất** trong `O(log n)`. Tuple được so theo phần tử đầu trước.

In [5]:
h = []
for item in [(5, "hồ 1"), (4, "hồ 2"), (9, "hồ 7")]:
    heapq.heappush(h, item)
print([heapq.heappop(h) for _ in range(3)])

[(4, 'hồ 2'), (5, 'hồ 1'), (9, 'hồ 7')]


### 3.4 Tính trước "lần mưa kế tiếp" bằng cách duyệt ngược

Duyệt từ cuối về đầu, giữ `seen[lake]` = ngày mưa gần nhất **phía sau**. Khi đó `nxt[i]` = lần tiếp theo trời mưa xuống hồ `rains[i]` (hoặc vô cực).

In [6]:
rains = [1, 2, 0, 0, 2, 1]
INF = float("inf")
nxt, seen = [INF] * len(rains), {}
for i in range(len(rains) - 1, -1, -1):
    if rains[i]:
        nxt[i] = seen.get(rains[i], INF)
        seen[rains[i]] = i
print(nxt)

[5, 4, inf, inf, inf, inf]


## 4. Ý tưởng nền tảng / trực giác

Làm tay **Ví dụ 2** `[1,2,0,0,2,1]`:

- Ngày 0, 1: hồ 1, 2 đầy. Ngày 2, 3 là ngày khô — nhưng **lúc này** ta chưa biết nên làm khô hồ nào! Nếu tham lam làm khô ngay hồ 1 ở ngày 2 thì vẫn ổn ở ví dụ này, nhưng nói chung có thể sai (xem Lỗi 1).
- Ý tưởng: **để dành** ngày khô vào "ví" `dry = [2, 3]`.
- Ngày 4: mưa hồ 2, mà hồ 2 đang đầy từ ngày 1 ⇒ cần một ngày khô **trong khoảng (1, 4)** ⇒ chọn ngày khô sớm nhất sau ngày 1 là **ngày 2** ⇒ `ans[2] = 2`. Ví còn `[3]`.
- Ngày 5: mưa hồ 1, đầy từ ngày 0 ⇒ cần ngày khô trong `(0, 5)` ⇒ ngày 3 ⇒ `ans[3] = 1`.
- Kết quả `[-1,-1,2,1,-1,-1]` ✔.

**Ví dụ 3** `[1,2,0,1,2]`: ví `[2]`. Ngày 3 mưa hồ 1 (đầy từ ngày 0) ⇒ dùng ngày 2. Ngày 4 mưa hồ 2 (đầy từ ngày 1) ⇒ cần ngày khô trong `(1, 4)` — ví rỗng ⇒ **không thể** ⇒ `[]` ✔.

Tại sao chọn ngày khô **sớm nhất** sau mốc? Ngày khô càng sớm càng "kén" (dùng được cho ít yêu cầu hơn, vì yêu cầu nào cũng cần ngày khô **sau** mốc của nó) ⇒ để ngày muộn lại cho các yêu cầu khác.

In [7]:
rains = [1, 2, 0, 0, 2, 1]
dry, last, ans = [], {}, [-1] * len(rains)
for i, r in enumerate(rains):
    if r == 0:
        dry.append(i); ans[i] = 1
        print(f"ngày {i}: khô -> cất vào ví {dry}")
        continue
    if r in last:
        j = bisect.bisect_right(dry, last[r])
        print(f"ngày {i}: mưa hồ {r} (đầy từ ngày {last[r]}) -> dùng ngày khô {dry[j]}")
        ans[dry.pop(j)] = r
    last[r] = i
print(ans)

ngày 2: khô -> cất vào ví [2]
ngày 3: khô -> cất vào ví [2, 3]
ngày 4: mưa hồ 2 (đầy từ ngày 1) -> dùng ngày khô 2
ngày 5: mưa hồ 1 (đầy từ ngày 0) -> dùng ngày khô 3
[-1, -1, 2, 1, -1, -1]


## 5. Cách 1 — Brute Force / baseline: quay lui thử mọi lựa chọn

**Ý tưởng:** mô phỏng từng ngày; ở mỗi ngày khô, **thử mọi hồ đang đầy** để làm khô (cộng thêm lựa chọn "làm khô một hồ rỗng" — tương đương không làm gì, ta ghi hồ `1` nếu hồ 1 đang rỗng, hoặc một hồ không bao giờ xuất hiện). Gặp ngày mưa vào hồ đầy ⇒ nhánh thất bại, quay lui.

Để "không làm gì" luôn hợp lệ, ta dùng hồ có số `max(rains) + 1` — hồ này không bao giờ bị mưa nên luôn rỗng.

In [8]:
def avoid_flood_brute(rains):
    n = len(rains)
    ans = [-1] * n
    dummy = max(rains) + 1                     # hồ không bao giờ mưa -> luôn rỗng
    def go(i, full):
        if i == n:
            return True
        r = rains[i]
        if r > 0:
            if r in full:
                return False
            return go(i + 1, full | {r})
        for lake in sorted(full) + [dummy]:     # thử từng lựa chọn
            ans[i] = lake
            if go(i + 1, full - {lake}):
                return True
        return False
    return ans[:] if go(0, frozenset()) else []

for rains, want in examples:
    got = avoid_flood_brute(rains)
    print(rains, "->", got, "| hợp lệ?" , is_valid(rains, got) if got else "(rỗng)")

[1, 2, 3, 4] -> [-1, -1, -1, -1] | hợp lệ? True
[1, 2, 0, 0, 2, 1] -> [-1, -1, 1, 2, -1, -1] | hợp lệ? True
[1, 2, 0, 1, 2] -> [] | hợp lệ? (rỗng)


**Độ phức tạp:** mỗi ngày khô có tới `(số hồ đầy + 1)` lựa chọn ⇒ hàm mũ, kiểu `O(k^z)` với `z` = số ngày khô. Chỉ dùng được với `n` rất nhỏ (≤ 12) — rất hữu ích để **kiểm chứng** khi nào đáp án là `[]`.

**Nút thắt:** ta quyết định "ngày khô này làm khô hồ nào" **quá sớm**, khi chưa biết tương lai ⇒ phải thử hết. Nếu **trì hoãn** quyết định tới lúc thật sự cần thì không phải đoán.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight (trì hoãn quyết định):** đi qua các ngày; ngày khô chỉ cất vào "ví". Khi trời mưa xuống hồ `L` **đang đầy** (đầy từ ngày `last[L]`), ta **bắt buộc** phải đã làm khô `L` vào một ngày khô nằm trong khoảng `(last[L], i)`. Lúc này mới chọn ngày khô cho `L`.

**Chọn ngày nào?** Ngày khô **sớm nhất** trong ví mà `> last[L]`.

**Lập luận trao đổi (exchange argument):** giả sử một lời giải tối ưu dùng ngày `d'` cho `L` trong khi ngày `d < d'` (vẫn `> last[L]`) còn trống hoặc được dùng cho hồ `M` với mốc `last[M] < d`. Đổi chéo `d ↔ d'`: `L` vẫn được làm khô trong `(last[L], i)`; `M` được ngày `d'` — vẫn trước hạn của `M`, vì các yêu cầu được xử lý theo thứ tự hạn `i` tăng dần, nên hạn của `M` ≥ `i` > `d'`. Vậy chọn sớm nhất không bao giờ làm mất lời giải.

**Quy tắc:**
- Nếu `rains[i] == 0` ⇒ thêm `i` vào ví, tạm đặt `ans[i] = 1`.
- Nếu mưa hồ `L` và `L` đang đầy ⇒ tìm ngày khô sớm nhất `> last[L]`; không có ⇒ trả `[]`; có ⇒ `ans[d] = L`, bỏ `d` khỏi ví.
- Luôn cập nhật `last[L] = i`.

Ngày khô **không** được dùng cho hồ nào: gán đại một hồ (ví dụ `1`) — làm khô hồ rỗng không đổi gì, còn nếu hồ 1 đang đầy thì làm khô nó cũng chẳng hại gì.

## 7. Các cách tiếp theo

### Cách 2 — Tham lam trì hoãn + `bisect` (ví là danh sách ngày khô tăng dần)

Ngày khô được thêm theo thứ tự tăng ⇒ `dry.append(i)` giữ danh sách luôn sắp xếp. Tìm bằng `bisect_right(dry, last[L])`, xoá bằng `dry.pop(j)`.

In [9]:
def avoid_flood_bisect(rains):
    ans = [-1] * len(rains)
    dry = []            # các ngày khô chưa dùng, tăng dần
    last = {}           # hồ đang đầy -> ngày bắt đầu đầy
    for i, r in enumerate(rains):
        if r == 0:
            dry.append(i)
            ans[i] = 1
            continue
        if r in last:
            j = bisect.bisect_right(dry, last[r])
            if j == len(dry):
                return []
            ans[dry.pop(j)] = r
        last[r] = i
    return ans

for rains, want in examples:
    got = avoid_flood_bisect(rains)
    print(got, is_valid(rains, got) if got else got == want)

[-1, -1, -1, -1] True
[-1, -1, 2, 1, -1, -1] True
[] True


Ở đây `last` chỉ cần chứa hồ **đang đầy**: khi ta làm khô `r` bằng ngày `d`, ngay sau đó trời lại mưa xuống `r` nên `r` lại đầy, và `last[r] = i` là đúng. Vì vậy `r in last` tương đương "`r` đang đầy".

Code nộp LeetCode (Cách 2):

In [10]:
class SolutionBisect:
    def avoidFlood(self, rains: list[int]) -> list[int]:
        ans = [-1] * len(rains)
        dry = []
        last = {}
        for i, r in enumerate(rains):
            if r == 0:
                dry.append(i)
                ans[i] = 1
                continue
            if r in last:
                j = bisect.bisect_right(dry, last[r])
                if j == len(dry):
                    return []
                ans[dry.pop(j)] = r
            last[r] = i
        return ans

print(SolutionBisect().avoidFlood([1, 2, 0, 0, 2, 1]))

[-1, -1, 2, 1, -1, -1]


**Độ phức tạp:** `bisect` là `O(log n)`, nhưng `list.pop(j)` dịch các phần tử phía sau ⇒ tệ nhất `O(n)` mỗi lần ⇒ `O(n²)` lý thuyết. Trên thực tế phép dịch là `memmove` rất nhanh nên vẫn chạy tốt với `n = 10^5` (có thể dùng `SortedList` của thư viện ngoài để đạt `O(n log n)`). Bộ nhớ `O(n)`.

### Cách 3 — Heap theo hạn chót (Earliest Deadline First)

Nhìn theo hướng ngược lại: **tại ngày khô**, ta quyết định ngay — nhưng quyết định **thông minh** nhờ biết trước tương lai. Tính trước `nxt[i]` = ngày kế tiếp trời mưa xuống hồ `rains[i]`. Khi hồ `L` đầy ở ngày `i` và sẽ bị mưa lại ở `nxt[i]`, đẩy `(nxt[i], L)` vào min-heap. Tới ngày khô: làm khô hồ có **hạn gần nhất** (đỉnh heap). Gặp mưa xuống hồ còn đầy ⇒ `[]`.

In [11]:
def avoid_flood_heap(rains):
    n = len(rains)
    INF = n
    nxt, seen = [INF] * n, {}
    for i in range(n - 1, -1, -1):
        if rains[i]:
            nxt[i] = seen.get(rains[i], INF)
            seen[rains[i]] = i
    ans = [-1] * n
    full = set()
    heap = []                       # (hạn chót, hồ)
    for i, r in enumerate(rains):
        if r:
            if r in full:
                return []
            full.add(r)
            if nxt[i] < INF:
                heapq.heappush(heap, (nxt[i], r))
        elif heap:
            _, lake = heapq.heappop(heap)
            full.discard(lake)
            ans[i] = lake
        else:
            ans[i] = 1
    return ans

for rains, want in examples:
    got = avoid_flood_heap(rains)
    print(got, is_valid(rains, got) if got else got == want)

[-1, -1, -1, -1] True
[-1, -1, 2, 1, -1, -1] True
[] True


Code nộp LeetCode (Cách 3):

In [12]:
class SolutionHeap:
    def avoidFlood(self, rains: list[int]) -> list[int]:
        n = len(rains)
        nxt, seen = [n] * n, {}
        for i in range(n - 1, -1, -1):
            if rains[i]:
                nxt[i] = seen.get(rains[i], n)
                seen[rains[i]] = i
        ans = [-1] * n
        full = set()
        heap = []
        for i, r in enumerate(rains):
            if r:
                if r in full:
                    return []
                full.add(r)
                if nxt[i] < n:
                    heapq.heappush(heap, (nxt[i], r))
            elif heap:
                _, lake = heapq.heappop(heap)
                full.discard(lake)
                ans[i] = lake
            else:
                ans[i] = 1
        return ans

print(SolutionHeap().avoidFlood([1, 2, 0, 0, 2, 1]))

[-1, -1, 2, 1, -1, -1]


**Độ phức tạp:** `O(n log n)` đảm bảo (mỗi hồ-lần-đầy vào/ra heap tối đa một lần), bộ nhớ `O(n)`.

Vì sao EDF đúng: tại ngày khô, mọi hồ trong heap đều cần được làm khô **trước hạn của nó**; làm khô hồ có hạn gần nhất là lựa chọn "khẩn cấp nhất" — lập luận trao đổi giống bài lập lịch kinh điển: nếu lời giải nào đó làm khô hồ hạn xa hơn hôm nay và để hồ hạn gần hơn vào ngày sau, đổi chéo vẫn hợp lệ.

### Cách 4 — DSU "ngày khô trống kế tiếp"

Thay `bisect + pop`, dùng mảng `parent` kiểu Union-Find: `find(x)` = ngày khô **chưa dùng** nhỏ nhất `≥ x`. Ngày mưa và ngày khô đã dùng trỏ sang `x + 1`. Nén đường đi giúp mỗi truy vấn gần `O(1)` (khấu hao `O(α(n))`).

In [13]:
def avoid_flood_dsu(rains):
    n = len(rains)
    parent = [i if (i == n or rains[i] == 0) else i + 1 for i in range(n + 1)]   # parent[n] = n: lính canh
    def find(x):
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:               # nén đường đi
            parent[x], x = root, parent[x]
        return root
    ans = [-1 if r else 1 for r in rains]
    last = {}
    for i, r in enumerate(rains):
        if r == 0:
            continue
        if r in last:
            d = find(last[r] + 1)
            if d >= i:
                return []
            ans[d] = r
            parent[d] = d + 1                   # đánh dấu đã dùng
        last[r] = i
    return ans

class SolutionDSU:
    def avoidFlood(self, rains: list[int]) -> list[int]:
        return avoid_flood_dsu(rains)

for rains, want in examples:
    got = avoid_flood_dsu(rains)
    print(got, is_valid(rains, got) if got else got == want)

[-1, -1, -1, -1] True
[-1, -1, 2, 1, -1, -1] True
[] True


**Độ phức tạp:** `O(n · α(n))` ≈ tuyến tính, bộ nhớ `O(n)`. Chú ý điều kiện `d >= i`: ngày khô tìm được phải **trước** ngày mưa hiện tại; ngày khô sau hôm nay vô dụng.

## 8. TRACE từng bước

Trace Cách 2 trên `rains = [1, 0, 2, 0, 1, 0, 2, 3, 3]` (tự dựng để có nhiều tình huống, kiểm chứng bằng `is_valid` và vét cạn).

| ngày | rains | ví `dry` trước | `last` trước | hành động | `ans` thay đổi |
|---|---|---|---|---|---|
| 0 | 1 | [] | {} | hồ 1 đầy | |
| 1 | 0 | [] | {1:0} | cất ngày 1 | ans[1]=1 tạm |
| 2 | 2 | [1] | {1:0} | hồ 2 đầy | |
| 3 | 0 | [1] | … | cất ngày 3 | |
| 4 | 1 | [1,3] | {1:0,2:2} | 1 đang đầy từ 0 ⇒ ngày khô sớm nhất > 0 là **1** | ans[1]=1 |
| 5 | 0 | [3] | {1:4,2:2} | cất ngày 5 | |
| 6 | 2 | [3,5] | | 2 đầy từ 2 ⇒ ngày khô > 2 là **3** | ans[3]=2 |
| 7 | 3 | [5] | | hồ 3 đầy | |
| 8 | 3 | [5] | {…,3:7} | 3 đầy từ 7 ⇒ cần ngày khô > 7 ⇒ **không có** ⇒ `[]` | |

In [14]:
def trace_bisect(rains):
    ans = [-1] * len(rains)
    dry, last = [], {}
    for i, r in enumerate(rains):
        if r == 0:
            dry.append(i); ans[i] = 1
            print(f"ngày {i}: khô  | ví={dry} last={last}")
            continue
        if r in last:
            j = bisect.bisect_right(dry, last[r])
            if j == len(dry):
                print(f"ngày {i}: mưa hồ {r}, đầy từ ngày {last[r]}, ví={dry} -> KHÔNG có ngày khô sau {last[r]} => []")
                return []
            d = dry.pop(j); ans[d] = r
            print(f"ngày {i}: mưa hồ {r}, đầy từ ngày {last[r]} -> dùng ngày khô {d} | ví={dry}")
        else:
            print(f"ngày {i}: mưa hồ {r} (đang rỗng)")
        last[r] = i
    print("ans =", ans)
    return ans

r = trace_bisect([1, 0, 2, 0, 1, 0, 2, 3, 3])
print("vét cạn xác nhận:", avoid_flood_brute([1, 0, 2, 0, 1, 0, 2, 3, 3]))

ngày 0: mưa hồ 1 (đang rỗng)
ngày 1: khô  | ví=[1] last={1: 0}
ngày 2: mưa hồ 2 (đang rỗng)
ngày 3: khô  | ví=[1, 3] last={1: 0, 2: 2}
ngày 4: mưa hồ 1, đầy từ ngày 0 -> dùng ngày khô 1 | ví=[3]
ngày 5: khô  | ví=[3, 5] last={1: 4, 2: 2}
ngày 6: mưa hồ 2, đầy từ ngày 2 -> dùng ngày khô 3 | ví=[5]
ngày 7: mưa hồ 3 (đang rỗng)
ngày 8: mưa hồ 3, đầy từ ngày 7, ví=[5] -> KHÔNG có ngày khô sau 7 => []
vét cạn xác nhận: []


Đổi một chút: chèn một ngày khô giữa hai lần mưa hồ 3 ⇒ có lời giải:

In [15]:
rains_ok = [1, 0, 2, 0, 1, 0, 2, 3, 0, 3]
r = trace_bisect(rains_ok)
print("hợp lệ?", is_valid(rains_ok, r))

ngày 0: mưa hồ 1 (đang rỗng)
ngày 1: khô  | ví=[1] last={1: 0}
ngày 2: mưa hồ 2 (đang rỗng)
ngày 3: khô  | ví=[1, 3] last={1: 0, 2: 2}
ngày 4: mưa hồ 1, đầy từ ngày 0 -> dùng ngày khô 1 | ví=[3]
ngày 5: khô  | ví=[3, 5] last={1: 4, 2: 2}
ngày 6: mưa hồ 2, đầy từ ngày 2 -> dùng ngày khô 3 | ví=[5]
ngày 7: mưa hồ 3 (đang rỗng)
ngày 8: khô  | ví=[5, 8] last={1: 4, 2: 6, 3: 7}
ngày 9: mưa hồ 3, đầy từ ngày 7 -> dùng ngày khô 8 | ví=[5]
ans = [-1, 1, -1, 2, -1, 1, -1, -1, 3, -1]
hợp lệ? True


Trace Cách 3 (heap) trên cùng input để thấy heap thay đổi:

In [16]:
def trace_heap(rains):
    n = len(rains)
    nxt, seen = [n] * n, {}
    for i in range(n - 1, -1, -1):
        if rains[i]:
            nxt[i] = seen.get(rains[i], n); seen[rains[i]] = i
    print("nxt =", nxt)
    ans, full, heap = [-1] * n, set(), []
    for i, r in enumerate(rains):
        if r:
            if r in full:
                print(f"ngày {i}: mưa hồ {r} đang đầy -> LỤT => []"); return []
            full.add(r)
            if nxt[i] < n: heapq.heappush(heap, (nxt[i], r))
            print(f"ngày {i}: mưa hồ {r} | heap={sorted(heap)}")
        elif heap:
            dl, lake = heapq.heappop(heap); full.discard(lake); ans[i] = lake
            print(f"ngày {i}: khô, làm khô hồ {lake} (hạn ngày {dl}) | heap={sorted(heap)}")
        else:
            ans[i] = 1
            print(f"ngày {i}: khô, không hồ nào cần -> ghi 1")
    print("ans =", ans)
    return ans

r = trace_heap(rains_ok)
print("hợp lệ?", is_valid(rains_ok, r))

nxt = [4, 10, 6, 10, 10, 10, 10, 9, 10, 10]
ngày 0: mưa hồ 1 | heap=[(4, 1)]
ngày 1: khô, làm khô hồ 1 (hạn ngày 4) | heap=[]
ngày 2: mưa hồ 2 | heap=[(6, 2)]
ngày 3: khô, làm khô hồ 2 (hạn ngày 6) | heap=[]
ngày 4: mưa hồ 1 | heap=[]
ngày 5: khô, không hồ nào cần -> ghi 1
ngày 6: mưa hồ 2 | heap=[]
ngày 7: mưa hồ 3 | heap=[(9, 3)]
ngày 8: khô, làm khô hồ 3 (hạn ngày 9) | heap=[]
ngày 9: mưa hồ 3 | heap=[]
ans = [-1, 1, -1, 2, -1, 1, -1, -1, 3, -1]
hợp lệ? True


Hai cách cho cùng kết quả ở đây, nhưng nói chung có thể khác nhau (đều hợp lệ).

## 9. Code nộp LeetCode

Bản chính: Cách 2 (tham lam trì hoãn + `bisect`) — ngắn, dễ giải thích.

In [17]:
from bisect import bisect_right

class Solution:
    def avoidFlood(self, rains: list[int]) -> list[int]:
        ans = [-1] * len(rains)
        dry = []                     # ngày khô chưa dùng (tăng dần)
        last = {}                    # hồ đang đầy -> ngày bắt đầu đầy
        for i, lake in enumerate(rains):
            if lake == 0:
                dry.append(i)
                ans[i] = 1           # mặc định: làm khô hồ 1 (vô hại)
                continue
            if lake in last:
                j = bisect_right(dry, last[lake])
                if j == len(dry):
                    return []
                ans[dry.pop(j)] = lake
            last[lake] = i
        return ans

for rains, want in examples:
    got = Solution().avoidFlood(rains)
    assert (got == [] and want == []) or is_valid(rains, got)
print("OK ví dụ đề")

OK ví dụ đề


## 10. Vì sao thuật toán đúng

**(a) Khi thuật toán trả mảng ⇒ mảng hợp lệ.** Mỗi lần trời mưa xuống hồ `L` đang đầy (từ ngày `p = last[L]`), ta gán một ngày khô `d` với `p < d < i` (vì `d` nằm trong ví ⇒ `d < i`, và `bisect_right` ⇒ `d > p`) cho `L`. Giữa `p` và `i` không có mưa nào khác xuống `L` (vì `p` là lần mưa gần nhất), nên ở ngày `d` hồ `L` đang đầy và được làm khô ⇒ ngày `i` không lụt. Mỗi ngày khô gán cho tối đa một hồ. Ngày khô còn lại ghi `1`: không gây lụt (làm khô chỉ giảm nước).

**(b) Khi thuật toán trả `[]` ⇒ thật sự không thể.** Ta xử lý các "yêu cầu" theo thứ tự hạn chót `i` tăng dần; mỗi yêu cầu `(p, i)` cần một ngày khô riêng trong `(p, i)`. Bằng lập luận trao đổi (mục 6), luôn tồn tại lời giải tối ưu dùng đúng các lựa chọn "ngày khô sớm nhất `> p`" của thuật toán. Nếu tới yêu cầu `(p, i)` mà không còn ngày khô nào trong `(p, i)` chưa dùng, thì mọi ngày khô trong khoảng đó đều đã bị chiếm bởi các yêu cầu có hạn `≤ i` — và lời giải tối ưu (đã được đổi về giống thuật toán) cũng không thể có ngày khô cho yêu cầu này ⇒ mâu thuẫn ⇒ không có lời giải.

Bộ test mục 12 kiểm chứng (b) bằng vét cạn trên hàng nghìn input nhỏ.

## 11. Phân tích Time / Space

Gọi `n = len(rains)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Quay lui | hàm mũ | `O(n)` độ sâu | chỉ để kiểm chứng |
| 2. Bisect + list | `O(n log n)` tìm kiếm + `O(n²)` tệ nhất cho `pop` | `O(n)` | thực tế nhanh (memmove) |
| 3. Heap EDF | `O(n log n)` | `O(n)` | đảm bảo lý thuyết |
| 4. DSU | `O(n · α(n))` | `O(n)` | gần tuyến tính |

Số hiệu hồ tới `10^9` ⇒ **không** dùng mảng đánh dấu theo hồ, phải dùng `dict/set`.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

Với mỗi cách: nếu đáp án mong đợi là `[]` thì phải trả `[]`; ngược lại kết quả phải **hợp lệ** theo `is_valid`.

In [18]:
ALL = {
    "brute": avoid_flood_brute,
    "bisect": avoid_flood_bisect,
    "heap": avoid_flood_heap,
    "dsu": avoid_flood_dsu,
    "sub_bisect": lambda a: SolutionBisect().avoidFlood(a),
    "sub_heap": lambda a: SolutionHeap().avoidFlood(a),
    "sub_dsu": lambda a: SolutionDSU().avoidFlood(a),
    "submit": lambda a: Solution().avoidFlood(a),
}
edge = [
    ([0], True), ([5], True), ([0, 0, 0], True),
    ([1, 1], False),                 # mưa liền hai ngày
    ([1, 0, 1], True),
    ([0, 1, 1], False),              # ngày khô ở TRƯỚC lần đầy đầu tiên -> vô dụng
    ([1, 0, 0, 1, 1], False),
    ([10**9, 0, 10**9], True),       # số hồ rất lớn
    ([1, 2, 0, 2, 3, 0, 1], True),
    ([69, 0, 0, 0, 69], True),
    ([1, 2, 0, 2, 0, 1], True),      # tham lam "làm khô hồ đầy sớm nhất" sẽ sai
    ([1, 0, 2, 0, 2, 1], True),      # dùng ngày khô trước mốc đầy sẽ sai
]
cases = [(r, w != []) for r, w in examples] + edge
for rains, feasible in cases:
    assert (avoid_flood_brute(rains) != []) == feasible, rains
for name, f in ALL.items():
    for rains, feasible in cases:
        got = f(rains)
        if feasible:
            assert is_valid(rains, got), (name, rains, got)
        else:
            assert got == [], (name, rains, got)
print("Mọi cách đúng với", len(cases), "test (khả thi hay không đã kiểm chứng bằng vét cạn)")

Mọi cách đúng với 15 test (khả thi hay không đã kiểm chứng bằng vét cạn)


### 12.2 Ngẫu nhiên, đối chiếu vét cạn (seed cố định)

`n ≤ 11`, số hồ `1..4`, khoảng 35% ngày khô ⇒ nhiều ca vừa khả thi vừa không khả thi.

In [19]:
rng = random.Random(1488)
stat = {True: 0, False: 0}
for _ in range(2000):
    n = rng.randint(1, 11)
    rains = [0 if rng.random() < 0.35 else rng.randint(1, 4) for _ in range(n)]
    feasible = avoid_flood_brute(rains) != []
    stat[feasible] += 1
    for name, f in ALL.items():
        if name == "brute":
            continue
        got = f(rains)
        if feasible:
            assert is_valid(rains, got), (name, rains, got)
        else:
            assert got == [], (name, rains, got)
print("2000 test ngẫu nhiên khớp vét cạn | khả thi:", stat[True], "| không khả thi:", stat[False])

2000 test ngẫu nhiên khớp vét cạn | khả thi: 1007 | không khả thi: 993


### 12.3 Dữ liệu lớn `n = 10^5`

Sinh dữ liệu **chắc chắn khả thi**: mỗi khi định mưa lại xuống một hồ đang đầy, chèn trước đó một ngày khô. Thêm một test xấu cho `list.pop`: nhiều ngày khô ở đầu, rồi lần lượt dùng chúng.

In [20]:
def feasible_big(rng, n):
    out, full = [], set()
    while len(out) < n:
        lake = rng.randint(1, 1000)
        if lake in full and rng.random() < 0.5:
            out.append(0); full.discard(lake)
        if lake not in full:
            out.append(lake); full.add(lake)
        else:
            out.append(0)
    return out[:n]

big1 = feasible_big(rng, 10**5)
half = 5 * 10**4 // 2
big2 = list(range(1, half + 1)) + [0] * half + list(range(1, half + 1))
for label, big in [("ngẫu nhiên", big1), ("dồn ngày khô", big2)]:
    for name in ["bisect", "heap", "dsu", "submit"]:
        t0 = time.perf_counter(); got = ALL[name](big); dt = time.perf_counter() - t0
        assert got and is_valid(big, got), (label, name)
        print(f"{label:13} | {name:7}: {dt:.3f}s")

ngẫu nhiên    | bisect : 0.024s


ngẫu nhiên    | heap   : 0.030s


ngẫu nhiên    | dsu    : 0.030s


ngẫu nhiên    | submit : 0.029s
dồn ngày khô  | bisect : 0.067s


dồn ngày khô  | heap   : 0.032s


dồn ngày khô  | dsu    : 0.022s


dồn ngày khô  | submit : 0.076s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quyết định ngay ở ngày khô một cách "mù"

Ở ngày khô, làm khô ngay **hồ đầy lâu nhất** (hoặc bất kỳ hồ đầy nào) mà không nhìn tương lai.

In [21]:
def wrong1(rains):
    ans, full = [], []                   # full giữ thứ tự bị đầy
    for r in rains:
        if r:
            if r in full:
                return []
            full.append(r); ans.append(-1)
        else:
            if full:
                ans.append(full.pop(0))   # SAI: làm khô hồ đầy sớm nhất, bất kể tương lai
            else:
                ans.append(1)
    return ans

case = [1, 2, 0, 2, 0, 1]
print("sai :", wrong1(case), "(trả [] dù có lời giải)")
good = Solution().avoidFlood(case)
print("đúng:", good, "| hợp lệ?", is_valid(case, good))

sai : [] (trả [] dù có lời giải)
đúng: [-1, -1, 2, -1, 1, -1] | hợp lệ? True


Với `[1,2,0,2,0,1]`: ngày khô đầu tiên (ngày 2) nên dành cho hồ **sắp bị mưa trước**, không phải hồ đầy trước. **Sửa:** trì hoãn quyết định (Cách 2) hoặc nhìn hạn chót (Cách 3).

### Lỗi 2 — Dùng ngày khô **trước** lúc hồ bắt đầu đầy

Lấy ngày khô sớm nhất trong ví mà quên điều kiện `> last[L]`. Làm khô một hồ đang **rỗng** thì vô ích.

In [22]:
def wrong2(rains):
    ans, dry, last = [-1] * len(rains), [], {}
    for i, r in enumerate(rains):
        if r == 0:
            dry.append(i); ans[i] = 1; continue
        if r in last:
            if not dry:
                return []
            ans[dry.pop(0)] = r           # SAI: không kiểm tra ngày khô > last[r]
        last[r] = i
    return ans

for case in [[0, 1, 1], [1, 0, 2, 0, 2, 1]]:
    bad = wrong2(case)
    print(case, "| sai:", bad, "hợp lệ?", is_valid(case, bad) if bad else "-",
          "| đúng:", Solution().avoidFlood(case))

[0, 1, 1] | sai: [1, -1, -1] hợp lệ? False | đúng: []
[1, 0, 2, 0, 2, 1] | sai: [-1, 2, -1, 1, -1, -1] hợp lệ? False | đúng: [-1, 1, -1, 2, -1, -1]


`[0,1,1]`: ngày khô duy nhất ở **trước** khi hồ 1 đầy ⇒ không cứu được ⇒ đáp án đúng là `[]`, bản sai lại trả một mảng (gây lụt). `[1,0,2,0,2,1]`: bản sai dùng ngày 1 (trước khi hồ 2 đầy ở ngày 2) cho hồ 2 ⇒ lụt ở ngày 4, dù thật ra có lời giải. **Sửa:** `bisect_right(dry, last[r])`.

### Lỗi 3 — Quên cập nhật `last[r] = i` sau khi dùng ngày khô

Nếu chỉ gán `last` ở lần mưa đầu, mốc "bắt đầu đầy" bị cũ ⇒ có thể chọn ngày khô nằm giữa hai lần mưa cũ, không cứu được lần mưa mới.

In [23]:
def wrong3(rains):
    ans, dry, last = [-1] * len(rains), [], {}
    for i, r in enumerate(rains):
        if r == 0:
            dry.append(i); ans[i] = 1; continue
        if r in last:
            j = bisect.bisect_right(dry, last[r])
            if j == len(dry):
                return []
            ans[dry.pop(j)] = r
        else:
            last[r] = i                   # SAI: chỉ ghi lần đầu
    return ans

case = [1, 0, 0, 1, 1]
bad = wrong3(case)
print("sai :", bad, "| hợp lệ?", is_valid(case, bad))
print("đúng:", Solution().avoidFlood(case), "| vét cạn:", avoid_flood_brute(case))

sai : [-1, 1, 1, -1, -1] | hợp lệ? False
đúng: [] | vét cạn: []


### Lỗi 4 — Để `0` ở ngày khô không dùng tới

Khởi tạo `ans = [0] * n` rồi chỉ ghi các ngày khô được dùng. Đề yêu cầu ngày khô phải **chọn một hồ** — hồ `0` không tồn tại (hồ đánh số từ 1). Hàm kiểm tra của ta (và LeetCode) không chấp nhận.

In [24]:
def wrong4(rains):
    ans, dry, last = [0] * len(rains), [], {}   # SAI: ngày khô thừa để 0
    for i, r in enumerate(rains):
        if r == 0:
            dry.append(i); continue
        ans[i] = -1
        if r in last:
            j = bisect.bisect_right(dry, last[r])
            if j == len(dry):
                return []
            ans[dry.pop(j)] = r
        last[r] = i
    return ans

case = [1, 0, 0, 1]
print("sai :", wrong4(case), "| hợp lệ?", is_valid(case, wrong4(case)))
print("đúng:", Solution().avoidFlood(case), "| hợp lệ?", is_valid(case, Solution().avoidFlood(case)))

sai : [-1, 1, 0, -1] | hợp lệ? False
đúng: [-1, 1, 1, -1] | hợp lệ? True


**Sửa:** gán mặc định `ans[i] = 1` cho mọi ngày khô.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tài nguyên xuất hiện theo thời gian, yêu cầu tới sau và cần một tài nguyên **trong khoảng** `(mốc, hạn)`" ⇒ **trì hoãn** và gán **tài nguyên sớm nhất thoả mãn** (bisect / DSU / SortedList).
- Hoặc biết trước tương lai ⇒ **EDF**: tại mỗi tài nguyên, phục vụ yêu cầu có hạn sớm nhất (heap).
- Bài nhiều đáp án ⇒ viết `is_valid` mô phỏng trước, test bằng nó.
- Khoá rất lớn (`10^9`) ⇒ `dict/set`, không mảng.
- Mẫu DSU "phần tử trống kế tiếp" cũng dùng ở: LC 1562, đặt chỗ ngồi, cấp phòng khách sạn, "Booking Concert Tickets".

## 15. Biến thể của bài

1. **Mỗi ngày khô làm khô được `c` hồ:** ví chứa mỗi ngày khô kèm "số lượt còn lại"; chỉ `pop` khi hết lượt.
2. **Hỏi số lần lụt tối thiểu** (cho phép lụt): bài trở thành chọn tập yêu cầu lớn nhất phục vụ được — vẫn tham lam theo hạn chót nhưng khi không còn ngày khô thì tính một lần lụt và tiếp tục (hồ vẫn đầy).
3. **Chỉ hỏi có tránh được không (True/False):** chính là kiểm tra `ans != []`.
4. **Bài lập lịch có hạn chót** (LC 630 Course Schedule III, LC 1353 Maximum Number of Events): cùng họ EDF + heap.
5. **LC 1562 / LC 2251**: kỹ thuật "tìm phần tử trống kế tiếp" bằng DSU hoặc sort + bisect.

In [25]:
# Minh hoạ biến thể 2: số lần lụt tối thiểu (cho phép lụt, hồ vẫn đầy sau khi lụt)
def min_floods(rains):
    dry, last, floods = [], {}, 0
    for i, r in enumerate(rains):
        if r == 0:
            dry.append(i); continue
        if r in last:
            j = bisect.bisect_right(dry, last[r])
            if j == len(dry):
                floods += 1
            else:
                dry.pop(j)
        last[r] = i
    return floods

print(min_floods([1, 2, 0, 1, 2]), min_floods([1, 1, 1]), min_floods([1, 2, 0, 0, 2, 1]))

1 2 0


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quay lui | Thử mọi hồ ở mỗi ngày khô | hàm mũ | `O(n)` | Kiểm chứng `[]` với `n` nhỏ |
| 2. Trì hoãn + bisect | Cất ngày khô, dùng ngày sớm nhất `> last[L]` khi cần | `O(n log n)`~`O(n²)` tệ nhất | `O(n)` | Bản nộp gọn, phổ biến |
| 3. Heap EDF | Biết trước lần mưa kế; ngày khô làm khô hồ hạn gần nhất | `O(n log n)` | `O(n)` | Muốn đảm bảo `O(n log n)` |
| 4. DSU ngày trống | `find(x)` = ngày khô trống đầu tiên `≥ x` | `O(n α(n))` | `O(n)` | Tối ưu nhất, mẫu dùng lại nhiều |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().avoidFlood([2, 0, 0, 2, 3, 0, 3])` trả gì?

### Bài 2 (tính tay)
Với `[1, 2, 0, 0, 1, 2]`, chạy tay Cách 2: ví `dry` và `last` sau mỗi ngày; đáp án là gì?

### Bài 3 (điền TODO)
Hoàn thiện `can_avoid(rains) -> bool` chỉ trả `True/False` (không cần dựng `ans`).

### Bài 4 (tự viết lại)
Viết lại Cách 3 (heap) nhưng **không tính trước** `nxt` bằng mảng: dùng `dict` ánh xạ hồ → danh sách các ngày mưa (theo thứ tự) và con trỏ.

### Bài 5 (mini challenge)
Mỗi ngày khô được làm khô **tối đa 2 hồ**; `ans[i]` là **list** các hồ được làm khô (độ dài 2, có thể lặp/thừa). Viết `avoid_flood_two(rains)` và hàm kiểm tra tương ứng.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def can_avoid(rains):
    # TODO: ví ngày khô + last; trả False khi không tìm được ngày khô phù hợp
    pass

# Bài 4 — TODO
def avoid_flood_heap2(rains):
    # TODO: days[lake] = [các ngày mưa], ptr[lake] = vị trí hiện tại
    pass

# Bài 5 — TODO
def avoid_flood_two(rains):
    # TODO: ví lưu [ngày, số lượt còn lại]; mỗi ngày khô có 2 lượt
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ngày 0 hồ 2 đầy; ngày 1, 2 khô. (2) Ngày 3 mưa hồ 2 ⇒ dùng ngày khô sớm nhất > 0 là ngày 1. (3) Ngày 6 mưa hồ 3 (đầy từ ngày 4) ⇒ cần ngày khô > 4: ngày 5 ⇒ ngày 2 ghi 1.

**Bài 2.** (1) Sau ngày 3: `dry = [2, 3]`, `last = {1: 0, 2: 1}`. (2) Ngày 4: dùng ngày 2 cho hồ 1. (3) Ngày 5: dùng ngày 3 cho hồ 2 ⇒ `[-1,-1,1,2,-1,-1]`.

**Bài 3.** (1) Copy Cách 2. (2) Thay `return []` bằng `return False`. (3) Cuối cùng `return True`.

**Bài 4.** (1) `days[lake].append(i)` khi duyệt trước. (2) Khi hồ đầy ở lần thứ `p`, lần kế là `days[lake][p + 1]` nếu có. (3) `ptr[lake] += 1` mỗi lần mưa.

**Bài 5.** (1) Ví: list `[ngày, lượt]` tăng dần theo ngày. (2) Khi dùng: giảm lượt, chỉ `pop` khi lượt = 0; ghi hồ vào `ans[ngày]`. (3) Cuối cùng đệm `ans[ngày]` đủ 2 phần tử bằng `1`. Hàm kiểm tra: ngày khô `discard` từng hồ trong list.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
ans1 = Solution().avoidFlood([2, 0, 0, 2, 3, 0, 3])
print(ans1)
assert ans1 == [-1, 2, 1, -1, -1, 3, -1] and is_valid([2, 0, 0, 2, 3, 0, 3], ans1)

# Bài 2
ans2 = trace_bisect([1, 2, 0, 0, 1, 2])
assert ans2 == [-1, -1, 1, 2, -1, -1]

[-1, 2, 1, -1, -1, 3, -1]
ngày 0: mưa hồ 1 (đang rỗng)
ngày 1: mưa hồ 2 (đang rỗng)
ngày 2: khô  | ví=[2] last={1: 0, 2: 1}
ngày 3: khô  | ví=[2, 3] last={1: 0, 2: 1}
ngày 4: mưa hồ 1, đầy từ ngày 0 -> dùng ngày khô 2 | ví=[3]
ngày 5: mưa hồ 2, đầy từ ngày 1 -> dùng ngày khô 3 | ví=[]
ans = [-1, -1, 1, 2, -1, -1]


In [28]:
# Bài 3
def can_avoid_sol(rains):
    dry, last = [], {}
    for i, r in enumerate(rains):
        if r == 0:
            dry.append(i); continue
        if r in last:
            j = bisect.bisect_right(dry, last[r])
            if j == len(dry):
                return False
            dry.pop(j)
        last[r] = i
    return True

# Bài 4
def avoid_flood_heap2_sol(rains):
    days = {}
    for i, r in enumerate(rains):
        if r:
            days.setdefault(r, []).append(i)
    ptr = {r: 0 for r in days}
    ans, full, heap = [-1] * len(rains), set(), []
    for i, r in enumerate(rains):
        if r:
            if r in full:
                return []
            full.add(r)
            ptr[r] += 1
            if ptr[r] < len(days[r]):
                heapq.heappush(heap, (days[r][ptr[r]], r))
        elif heap:
            _, lake = heapq.heappop(heap)
            full.discard(lake); ans[i] = lake
        else:
            ans[i] = 1
    return ans

rng = random.Random(17)
for _ in range(800):
    rains = [0 if rng.random() < 0.35 else rng.randint(1, 4) for _ in range(rng.randint(1, 11))]
    feas = avoid_flood_brute(rains) != []
    assert can_avoid_sol(rains) == feas
    got = avoid_flood_heap2_sol(rains)
    assert (is_valid(rains, got) if feas else got == [])
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [29]:
# Bài 5
def avoid_flood_two_sol(rains):
    ans = [-1 if r else [] for r in rains]
    dry, last = [], {}                  # dry: list [ngày, lượt còn lại], tăng dần theo ngày
    for i, r in enumerate(rains):
        if r == 0:
            dry.append([i, 2]); continue
        if r in last:
            j = bisect.bisect_right(dry, [last[r], 10])
            if j == len(dry):
                return []
            d = dry[j]
            ans[d[0]].append(r)
            d[1] -= 1
            if d[1] == 0:
                dry.pop(j)
        last[r] = i
    for i, r in enumerate(rains):
        if r == 0:
            ans[i] += [1] * (2 - len(ans[i]))
    return ans

def is_valid_two(rains, ans):
    full = set()
    for r, a in zip(rains, ans):
        if r:
            if a != -1 or r in full:
                return False
            full.add(r)
        else:
            if len(a) != 2 or min(a) < 1:
                return False
            for x in a:
                full.discard(x)
    return True

def brute_two(rains):
    n = len(rains)
    def go(i, full):
        if i == n:
            return True
        r = rains[i]
        if r:
            return r not in full and go(i + 1, full | {r})
        opts = sorted(full)
        if len(opts) <= 2:
            return go(i + 1, frozenset())
        return any(go(i + 1, full - {a, b}) for a in opts for b in opts if a < b)
    return go(0, frozenset())

res5 = avoid_flood_two_sol([1, 2, 0, 1, 2])
print(res5)
assert is_valid_two([1, 2, 0, 1, 2], res5)
for _ in range(800):
    rains = [0 if rng.random() < 0.3 else rng.randint(1, 4) for _ in range(rng.randint(1, 11))]
    got = avoid_flood_two_sol(rains)
    if brute_two(rains):
        assert is_valid_two(rains, got), (rains, got)
    else:
        assert got == []
print("Đáp án Bài 1–5: mọi assert đều qua")

[-1, -1, [1, 2], -1, -1]
Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "trì hoãn + tài nguyên sớm nhất sau mốc":**

```python
ans = [-1] * n
dry, last = [], {}
for i, x in enumerate(rains):
    if x == 0:
        dry.append(i); ans[i] = 1           # cất vào ví, mặc định hồ 1
        continue
    if x in last:                           # hồ đang đầy
        j = bisect_right(dry, last[x])      # ngày khô sớm nhất > mốc
        if j == len(dry):
            return []
        ans[dry.pop(j)] = x
    last[x] = i                             # luôn cập nhật mốc
return ans
```

**Checklist trước khi submit:**
- [ ] Ngày khô dùng cho hồ `L` phải **sau** lần đầy gần nhất của `L` (`bisect_right(dry, last[L])`).
- [ ] Luôn cập nhật `last[L] = i` ở mỗi lần mưa.
- [ ] Ngày khô không dùng tới ghi `1` (không để `0`).
- [ ] Không lập mảng theo số hiệu hồ (tới `10^9`).
- [ ] Test các ca: `[0,1,1]`, `[1,1]`, `[1,0,2,0,2,1]`, test lớn `10^5`.
- [ ] Đúng chữ ký: `avoidFlood(self, rains: list[int]) -> list[int]`.